# 결정 트리 :
- 캔 와인 판매 : 특수캔으로 맛과 향을 유지하도록 제작
- 문제 : 레드/화이트 표시 누락
- 도수, 당도, pH 값으로 와인 종류 구별하는 방법?


## 로지스틱 회귀로 와인 분류하기 :
- 6,497개 샘플 데이터
- 화이트와인을 양성클래스(1) // 화이트(1), 레드(0) 중 선택

In [1]:
# 데이터셋 불러오기
import pandas as pd

wine = pd.read_csv('https://bit.ly/wine-date')

In [ ]:
wine.head()

In [ ]:
wine.info()
# 6496개 , 컬럼명, 널값, 자료형 확인

In [ ]:
wine.describe()
# 간략한 통계

In [ ]:
# 도수, 당도, pH 의 스케일이 다르다
# pH : 정수 한자리 // 알콜, 슈가 : 정수 1자리 ~ 두자리
# ==> StandardScaler 클래스의 사용하여 특성을 표준화


In [6]:
# 표준화 작업을 위해  데이터프레임을 넘파이로 바꾼다

# 데이터값들을 넘파이로 전환
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()

# 타깃값을 넘파이로 전환
target = wine['class'].to_numpy()

In [ ]:
# data 확인
data

In [ ]:
# target 확인
target

In [14]:
# 훈련세트와 테스트세트를 나눈다
# 보통은 test_size가 25% -> 20% 설정 ( 샘플갯수가 충분하므로 )

from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42)

In [ ]:
# 훈련세트와 테스트세트 크기 확인

print(train_input.shape, test_input.shape)

In [16]:
# 특성마다 스케일 다르므로 특성을 표준화
# StandardScaler 클래스 사용

from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(train_input)

train_scaled = ss.transform(train_input)  # 훈련세트 전처리
test_scaled = ss.transform(test_input)    # 테스트세트 전처리

# 모든 준비 끝!

In [ ]:
# 표준점수로 반환된 train_scaled, test_scaled 를 사용해 로지스틱 회귀모델을 훈련

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()                     # 객체 생성
lr.fit(train_scaled, train_target)            # 훈련

print(lr.score(train_scaled, train_target))   # 평가
print(lr.score(test_scaled, test_target))     # 평가

# 평가점수가 높지 않다
# 훈련, 테스트 둘다 낮다 -> 과소적합

## 설명하기 쉬운 모델과 어려운 모델

In [ ]:
# 위에서 로지스틱 회귀가 학습한 계수와 절편을 출력

print(lr.coef_, lr.intercept_)

# [[ 0.51268071  1.67335441 -0.68775646]] [1.81773456]
#    알콜도수     당도       pH           // 절편


In [ ]:
train_scaled

In [ ]:
train_target

In [ ]:
#  x = 도수 +  당도 +  pH  + 절편값
#  x =  (도수 *  0.51270274) + (당도* 1.6733911) + (pH * -0.68767781) +  (절편값 : 1.81777902 )

#  x > 0  ==> 화이트 와인
#  x < 0  ==> 레드 와인

# 도수, 당도 가 높을 수록  ( 곱하면 양수 ) --> 화이트 가능성 높고
# pH가 높을 수록 ( 곱하면 음수 ) --> 레드 가능성 높고

# 좀 쉬운 방법은 없을까 : 순서도처럼 직관적이고 간단한 방법 ?

## 결정 트리    
- 결정 트리 모델은 스무고개와 같다.
- 질문에 yes면 왼쪽, no면 오른쪽
- 질문의 퀼리티가 좋다면 분류 정확도도 높다
- 사이킷런에서 제공하는 결정트리 알고리즘 : DecisionTreeClassifier
- fit() 훈련, score() 정확도 평가  : 이전 모델들과 사용법 같다


In [ ]:
# 결정트리알고리즘을 사용하여 훈련
# 앞서 표준화한 train_scaled, test_scaled  사용

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)  # 모델 객체 생성
dt.fit(train_scaled, train_target)            # 훈련

print(dt.score(train_scaled, train_target))   # 훈련세트 평가
print(dt.score(test_scaled, test_target))     # 테스트세트 평가

# 우와 점수가 높다~
# 훈련세트 정확도 > 테스트세트 정확도  ==> 과대적합된 모델

# 뒤에 과대적합을 막는 가지치기 사용할 것임

In [ ]:
# 모델의 결과를 트리 모양으로 표현할 수 없을까?
# plot_tree() 결정트리를 그림으로 출력해 줌

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(10,7))   # 그림 사이즈
plot_tree(dt)
plt.show()                   # 시간이 많이 걸림

# 엄청난 트리
# 맨 위 노드 : 루트 노드 // 맨아래 노드 : 리프 노드
# 일반적으로 하나의 노드는 2개의 가지를 갖는다.

# 분류 : 루트부터 타고 내려가서 마지막 리프노드에 도착 --> 다수의 클래스가 예측 클래스
# 회귀 : 리프노드의 클래스들의 평균이 -> 예측 값

In [ ]:
# 너무 복잡하니 트리의 깊이를 제한해보자
# plt_tree 함수의 속성 지정
# max_depth = 1   --> 트리의 깊이 , 루트는 제외하고 하나의 노드를 더 확장
#     제한하지 않으면 리프노드가 순수노드가 될때까지 진행 ...
# filled = True   --> 클래스에 맞는 노드의 색 채움
# feature_names = [ '특성이름' ,'','']  --> 특성이름 전달

plt.figure(figsize=(10,7))
plot_tree(dt, max_depth=1, filled=True, feature_names=['alcohol', 'sugar', 'pH'])  # max_depth=2도  실행해 볼것
plt.show()


### 트리 그래프 설명   

- 루트 부분
  - sugar <= -0.239   --> 테스트 조건
  - 조건을 만족하면 왼쪽, 불만족하면 오른쪽
  - value = [ 1258, 3939 ]  --> 0-음성(레드) : 1258, 1-양성(화이트): 3939

- 왼쪽 노드
  - sugar <= -0.802   --> 테스트 조건
  - 조건을 만족하면 왼쪽, 불만족하면 오른쪽
  - value = [ 1177, 1745 ] --> 음성 , 양성
  - 두번째 화이트 와인 갯수 많이 줄어듬

- 오른쪽 노드
  - sugar <= 0.204  --> 테스트 조건
  - value = [ 81, 2194 ]  --> 화이트 와인이 많다.

- 노드마다 색깔이 다른다
  - filled = True 하면 양성 비율이 높을수록 색이 진하다.


### 불순도 : gini 불순도
- criterion 매개변수의 기본값 : criterion='gini'
- 데이터를 분할할 기준을 정하는 것
- 루트에서 조건 sugar <= -0.239 에서 -0.239 는 어떻게 나왔나?
- 지니 불순도 : 클래스의 비율을 제곱해서 더한 다음 1을 빼준다.
  - 1 -( (1258/5197)제곱 + (3939/5197)제곱 ) = 0.367
  - 우리가 찾는 것은 양성인데 그중 0.367 불순(다른것) 섞여있다.
- 노드에 하나의 클래스(양성이든 음성이드) --> 순수노드

- 부모와 자식의 불순도 차이 => 정보이득
- 정보이득이 최대가 되도록 데이터를 나눔
--------
- 부모노드와 자식노드의 불순도 차이가 큰쪽으로 분할을 해서 점점 더 좋은 예측을 만들도록 학습함   -->  DecisionTreeClassifier 가 학습

## 가지치기
- 결정트리의 최대깊이를 조정
- 훈련세트 정확도 > 데스트세트 정확도   :   과대적합 될 확률이 높다
- max_depth = 1   --> 트리의 깊이 , 루트는 제외
    -  제한하지 않으면 리프노드가 순수노드가 될때까지 진행 ...
- max_features = None  --> 특성의 갯수를 지정할 수 있다 / 기본은 None
- max_features =2 --> 랜덤하게 선택
- 조건 특성은 각 특성의 불순도를 계산하여 불순도가 큰것으로 선택 // 같으면 랜덤

In [ ]:
# 가지치기가 결정트리의 과대적합을 막는 방법 중 하나
# 대표적인 방법 --> max_depth 사용

dt = DecisionTreeClassifier(max_depth=3, random_state=42)  # max_depth 값 수정해볼 것
dt.fit(train_scaled, train_target)

print(dt.score(train_scaled, train_target))
print(dt.score(test_scaled, test_target))

In [ ]:
# 위 모델을 트리로 표현하자

plt.figure(figsize=(20,15))
plot_tree(dt, filled=True, feature_names=['alcohol', 'sugar', 'pH'])
plt.show()

# 루트노드 당도를 기준으로 분할
# depth1도 당도를 기준으로 훈련세트 분할
# depth2는 왼쪽은 당도기준, 오른쪽은 도수를 기준으로 분할 , 오른쪽 두 노드는 pH를 사용하여 분할

# 리프노드의 왼쪽에서 세번째만 음성클래스가 많다. 즉, 이 노드에 도착하면 레드와인으로 예측
# 레드와인의 기준을 설명할 것

In [ ]:
# 특성값의 스케일은 결정 트리 알고리즘에 아무런 영향을 미치지 않으므로 표준화 전처리를 할 필요가 없음
# 전처리하기 전의 훈련 세트(train_input )와 테스트 세트(test_input )로 결정 트리 모델을 다시 훈련

dt = DecisionTreeClassifier(max_depth=3, random_state=42)

dt.fit(train_input, train_target)

print(dt.score(train_input, train_target))
print(dt.score(test_input, test_target))

In [ ]:
# # 위 모델을 트리로 표현하자
plt.figure(figsize=(20,15))
plot_tree(dt, filled=True, feature_names=['alcohol', 'sugar', 'pH'])
plt.show()

In [ ]:
# 특성 중요도 출력

print(dt.feature_importances_ )

# 출력 결과 : [0.12345626 0.86862934 0.0079144 ]
# 알콜도수, 슈가, pH  --> 슈가가 중요 특성임을 알 수 있다.

### 결정트리 알고리즘
- 불순도를 기준으로 샘플을 나눈다
- 불순도는 클래스별의 비율로 계산
- 즉, 특성값의 스케일이 계산에 영향을 미치지 않는다 ==> 표준화 전처리 필요없음  ==> 결정트리 알고리즘의 장점 중 하나
- 고도화된 if트리라고 생각해도 좋다

--------------
- 트리 성장시키면 과대적합, 트리를 제한하면 성능이 안좋다
==> 결정트리 딜레마...!

- 트리 여러개를 가지고 앙상블을 만들 수 있다. 앙상블은 성능이 좋다!
------------
- 지도 학습 : 입력, 타깃
  - 분류 : 로지스틱, k-nn, 결정트리
  - 회규 : 선형회귀


## 실습 문제
붓꽃(Iris) 데이터로 결정트리를 학습한 후 다음을 수행하시오.

- 학습/테스트 데이터 분리
- 결정트리 객체 생성 (max_depth=3)
- 모델 학습
- 정확도 출력
- 특성 중요도 출력
- 결정트리 시각화

In [35]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# 0. 데이터 로드
iris = load_iris()

X = iris.data
Y = iris.target


In [ ]:
# 1. 학습/테스트 데이터 분리
train_input, test_input, train_target, test_target = train_test_split(
    ??, ??,  test_size=0.2, random_state=42 )


In [ ]:
# 2. 결정트리 객체 생성 : max_depth=3
dt = DecisionTreeClassifier(
     ??, ??  )


In [ ]:
# 4. 모델 학습
dt.?? (train_input, train_target)

# 5. 정확도 출력
score = dt.?? (test_input, test_target)
print("정확도 :", score )

In [ ]:
# 6. 특성 중요도 출력
print("특성 중요도")

for name, importance in zip( iris.feature_names, dt.feature_importances_):
    print(f"{name:20s} : {importance:.3f}")

In [ ]:
# 7. 결정트리 시각화
plt.figure(figsize=(12, 8))

plot_tree(
    dt,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)

plt.show()